In [2]:
import os
import random
import time
import itertools
import numpy as np
from collections import defaultdict

In [3]:
def load_documents(path="minhash"):
    docs = {}
    for i in range(1, 5):
        with open(os.path.join(path, f"D{i}.txt"), "r") as f:
            docs[f"D{i}"] = f.read().strip()
    return docs

documents = load_documents()
documents

{'D1': 'apple ceo tim cook is spending some time in canada this week and yesterday he attended a hockey game and visited the eaton centre apple store in toronto cook today stopped by the offices of canadian ecommerce platform shopify where he spoke to the financial post about augmented reality apps and the homepod on the topic of the homepod cook said that apples deep integration between hardware and software will help to differentiate the smart speaker from competing products like amazons alexa and the google home competition makes all of us better and i welcome it cook said but if you are both trying to license something and compete with your licensees this is a difficult model and it remains to be seen if it can be successful or not cook also said a quality very immersive audio experience was one thing missing from the smart speaker market which apple is aiming to fix music deserves that kind of quality as opposed to some kind of squeaky sound he said the homepod which at in the uni

## K-gram construction 

In [4]:
def char_kgrams(text, k):
    return set([text[i:i+k] for i in range(len(text)-k+1)])

def word_kgrams(text, k):
    words = text.split()
    return set([" ".join(words[i:i+k]) for i in range(len(words)-k+1)])

In [5]:
## biulding all required k-grams 
kgram_types = {
    "char_2": lambda text: char_kgrams(text, 2),
    "char_3": lambda text: char_kgrams(text, 3),
    "word_2": lambda text: word_kgrams(text, 2),
}

kgram_data = {}

for ktype in kgram_types:
    kgram_data[ktype] = {}
    for doc in documents:
        kgram_data[ktype][doc] = kgram_types[ktype](documents[doc])

## Exact Jaccard Similarity

In [6]:
def jaccard(set1, set2):
    return len(set1 & set2) / len(set1 | set2)

pairs = list(itertools.combinations(documents.keys(), 2))

exact_results = {}

for ktype in kgram_data:
    exact_results[ktype] = {}
    for d1, d2 in pairs:
        sim = jaccard(kgram_data[ktype][d1], kgram_data[ktype][d2])
        exact_results[ktype][(d1, d2)] = sim

exact_results

{'char_2': {('D1', 'D2'): 0.9811320754716981,
  ('D1', 'D3'): 0.8156996587030717,
  ('D1', 'D4'): 0.6444444444444445,
  ('D2', 'D3'): 0.8,
  ('D2', 'D4'): 0.6412698412698413,
  ('D3', 'D4'): 0.6529968454258676},
 'char_3': {('D1', 'D2'): 0.977979274611399,
  ('D1', 'D3'): 0.5803571428571429,
  ('D1', 'D4'): 0.3050847457627119,
  ('D2', 'D3'): 0.5680473372781065,
  ('D2', 'D4'): 0.30590339892665475,
  ('D3', 'D4'): 0.31212381771281167},
 'word_2': {('D1', 'D2'): 0.9407665505226481,
  ('D1', 'D3'): 0.18234165067178504,
  ('D1', 'D4'): 0.03024193548387097,
  ('D2', 'D3'): 0.1736641221374046,
  ('D2', 'D4'): 0.030303030303030304,
  ('D3', 'D4'): 0.01607142857142857}}

## MIN-HASHING IMPLEMENTATION

We use hash family: 
h(x)=(ax+b)mod m

In [7]:
def compute_minhash_signature(kgram_set, universe, hash_funcs, m):
    signature = []

    for a, b in hash_funcs:
        min_hash = float('inf')
        for item in kgram_set:
            x = universe[item]
            hash_val = (a * x + b) % m
            if hash_val < min_hash:
                min_hash = hash_val
        signature.append(min_hash)

    return signature

def build_universe(kgram_sets):
    universe = {}
    index = 0
    for s in kgram_sets.values():
        for item in s:
            if item not in universe:
                universe[item] = index
                index += 1
    return universe

##genrating hash functions
def generate_hash_functions(t, m):
    hash_funcs = []
    for _ in range(t):
        a = random.randint(1, m-1)
        b = random.randint(0, m-1)
        hash_funcs.append((a, b))
    return hash_funcs

## approximate Jaccard similarity
def approx_jaccard(sig1, sig2):
    count = sum(1 for i in range(len(sig1)) if sig1[i] == sig2[i])
    return count / len(sig1)

In [8]:
m = 20000
t_values = [20, 60, 150, 300, 600]

# Using 3-grams (as required)
kgrams_3 = kgram_data["char_3"]

universe = build_universe(kgrams_3)

results = {}

for t in t_values:
    hash_funcs = generate_hash_functions(t, m)

    sig_D1 = compute_minhash_signature(kgrams_3["D1"], universe, hash_funcs, m)
    sig_D2 = compute_minhash_signature(kgrams_3["D2"], universe, hash_funcs, m)

    sim = approx_jaccard(sig_D1, sig_D2)
    results[t] = sim

results

{20: 1.0,
 60: 0.95,
 150: 0.9933333333333333,
 300: 0.9666666666666667,
 600: 0.9716666666666667}

## LSH IMPLEMENTATION

We use:
𝑓(𝑠)=1−(1−𝑠^𝑟)^𝑏

In [9]:
def s_curve(s, r, b):
    return 1 - (1 - s**r)**b

##computing prob for all pairs using LSH parameters r and b
t = 160
r = 8
b = 20

hash_funcs = generate_hash_functions(t, m)

signatures = {}

for doc in documents:
    signatures[doc] = compute_minhash_signature(
        kgrams_3[doc], universe, hash_funcs, m
    )

lsh_probs = {}

for d1, d2 in pairs:
    est_sim = approx_jaccard(signatures[d1], signatures[d2])
    prob = s_curve(est_sim, r, b)
    lsh_probs[(d1, d2)] = prob

lsh_probs

{('D1', 'D2'): 0.9999999999998961,
 ('D1', 'D3'): 0.15479235805083957,
 ('D1', 'D4'): 0.004494128995338298,
 ('D2', 'D3'): 0.1093115481562198,
 ('D2', 'D4'): 0.004494128995338298,
 ('D3', 'D4'): 0.003891952113603825}

## MOVIELENS – EXACT JACCARD

In [11]:
def load_movielens(path="ml-100k/u.data"):
    user_movies = defaultdict(set)

    with open(path, "r") as f:
        for line in f:
            user, movie, rating, _ = line.strip().split("\t")
            user_movies[user].add(movie)

    return user_movies

user_movies = load_movielens()

In [12]:
users = list(user_movies.keys())

exact_pairs = []

for i in range(len(users)):
    for j in range(i+1, len(users)):
        sim = jaccard(user_movies[users[i]], user_movies[users[j]])
        if sim >= 0.5:
            exact_pairs.append((users[i], users[j]))

len(exact_pairs)

10

## MOVIELENS MINHASH EXPERIMENT

In [13]:
print("Preparing MovieLens MinHash Experiment...\n")

# Build universe for MovieLens (movies)
all_movies = set()
for movies in user_movies.values():
    all_movies.update(movies)

movie_universe = {movie: idx for idx, movie in enumerate(all_movies)}

m = 50000  # large enough prime space

def compute_user_signature(user_set, universe, hash_funcs, m):
    signature = []
    for a, b in hash_funcs:
        min_hash = float('inf')
        for movie in user_set:
            x = universe[movie]
            hash_val = (a * x + b) % m
            if hash_val < min_hash:
                min_hash = hash_val
        signature.append(min_hash)
    return signature


def run_minhash_experiment(num_hashes, threshold=0.5, runs=5):
    
    print(f"\nRunning for {num_hashes} hash functions...\n")
    
    false_positives_total = 0
    false_negatives_total = 0
    
    for run in range(runs):
        print(f"Run {run+1}")
        
        hash_funcs = generate_hash_functions(num_hashes, m)
        
        # Compute signatures
        signatures = {}
        for user in users:
            signatures[user] = compute_user_signature(
                user_movies[user],
                movie_universe,
                hash_funcs,
                m
            )
        
        estimated_pairs = set()
        exact_pairs_set = set()
        
        for i in range(len(users)):
            for j in range(i+1, len(users)):
                
                u1 = users[i]
                u2 = users[j]
                
                exact_sim = jaccard(user_movies[u1], user_movies[u2])
                approx_sim = approx_jaccard(signatures[u1], signatures[u2])
                
                if exact_sim >= threshold:
                    exact_pairs_set.add((u1, u2))
                
                if approx_sim >= threshold:
                    estimated_pairs.add((u1, u2))
        
        false_positives = len(estimated_pairs - exact_pairs_set)
        false_negatives = len(exact_pairs_set - estimated_pairs)
        
        false_positives_total += false_positives
        false_negatives_total += false_negatives
        
        print("False Positives:", false_positives)
        print("False Negatives:", false_negatives)
        print("-----")
    
    print("\nAverage over 5 runs:")
    print("Avg False Positives:", false_positives_total / runs)
    print("Avg False Negatives:", false_negatives_total / runs)


# Run experiments
run_minhash_experiment(50)
run_minhash_experiment(100)
run_minhash_experiment(200)

Preparing MovieLens MinHash Experiment...


Running for 50 hash functions...

Run 1
False Positives: 355
False Negatives: 2
-----
Run 2
False Positives: 65
False Negatives: 4
-----
Run 3
False Positives: 109
False Negatives: 3
-----
Run 4
False Positives: 145
False Negatives: 4
-----
Run 5
False Positives: 115
False Negatives: 3
-----

Average over 5 runs:
Avg False Positives: 157.8
Avg False Negatives: 3.2

Running for 100 hash functions...

Run 1
False Positives: 28
False Negatives: 1
-----
Run 2
False Positives: 36
False Negatives: 0
-----
Run 3
False Positives: 33
False Negatives: 4
-----
Run 4
False Positives: 25
False Negatives: 1
-----
Run 5
False Positives: 23
False Negatives: 2
-----

Average over 5 runs:
Avg False Positives: 29.0
Avg False Negatives: 1.6

Running for 200 hash functions...

Run 1
False Positives: 16
False Negatives: 3
-----
Run 2
False Positives: 7
False Negatives: 1
-----
Run 3
False Positives: 10
False Negatives: 4
-----
Run 4
False Positives: 10
False Negat

## LSH on MOVIELENS

In [14]:
def lsh_candidates(signatures, r, b):
    buckets = defaultdict(list)
    
    for user, sig in signatures.items():
        for band in range(b):
            start = band * r
            end = start + r
            band_tuple = tuple(sig[start:end])
            buckets[(band, band_tuple)].append(user)
    
    candidates = set()
    
    for bucket_users in buckets.values():
        if len(bucket_users) > 1:
            for pair in itertools.combinations(bucket_users, 2):
                candidates.add(tuple(sorted(pair)))
    
    return candidates


def run_lsh_experiment(num_hashes, r, b, threshold=0.6, runs=5):
    
    print(f"\nRunning LSH for {num_hashes} hashes | r={r}, b={b}")
    
    false_positives_total = 0
    false_negatives_total = 0
    
    for run in range(runs):
        print(f"Run {run+1}")
        
        hash_funcs = generate_hash_functions(num_hashes, m)
        
        signatures = {}
        for user in users:
            signatures[user] = compute_user_signature(
                user_movies[user],
                movie_universe,
                hash_funcs,
                m
            )
        
        candidates = lsh_candidates(signatures, r, b)
        
        exact_pairs_set = set()
        
        for i in range(len(users)):
            for j in range(i+1, len(users)):
                u1 = users[i]
                u2 = users[j]
                if jaccard(user_movies[u1], user_movies[u2]) >= threshold:
                    exact_pairs_set.add((u1, u2))
        
        # Evaluate only candidates
        estimated_pairs = set()
        for u1, u2 in candidates:
            approx_sim = approx_jaccard(signatures[u1], signatures[u2])
            if approx_sim >= threshold:
                estimated_pairs.add((u1, u2))
        
        false_positives = len(estimated_pairs - exact_pairs_set)
        false_negatives = len(exact_pairs_set - estimated_pairs)
        
        false_positives_total += false_positives
        false_negatives_total += false_negatives
        
        print("False Positives:", false_positives)
        print("False Negatives:", false_negatives)
        print("-----")
    
    print("\nAverage over 5 runs:")
    print("Avg False Positives:", false_positives_total / runs)
    print("Avg False Negatives:", false_negatives_total / runs)

## Experiments with different parameters

In [15]:
# For similarity >= 0.6

run_lsh_experiment(50, r=5, b=10, threshold=0.6)
run_lsh_experiment(100, r=5, b=20, threshold=0.6)
run_lsh_experiment(200, r=5, b=40, threshold=0.6)
run_lsh_experiment(200, r=10, b=20, threshold=0.6)

# For similarity >= 0.8

run_lsh_experiment(50, r=5, b=10, threshold=0.8)
run_lsh_experiment(100, r=5, b=20, threshold=0.8)
run_lsh_experiment(200, r=5, b=40, threshold=0.8)
run_lsh_experiment(200, r=10, b=20, threshold=0.8)


Running LSH for 50 hashes | r=5, b=10
Run 1
False Positives: 1
False Negatives: 0
-----
Run 2
False Positives: 1
False Negatives: 2
-----
Run 3
False Positives: 0
False Negatives: 1
-----
Run 4
False Positives: 6
False Negatives: 0
-----
Run 5
False Positives: 3
False Negatives: 1
-----

Average over 5 runs:
Avg False Positives: 2.2
Avg False Negatives: 0.8

Running LSH for 100 hashes | r=5, b=20
Run 1
False Positives: 0
False Negatives: 0
-----
Run 2
False Positives: 0
False Negatives: 1
-----
Run 3
False Positives: 0
False Negatives: 1
-----
Run 4
False Positives: 0
False Negatives: 0
-----
Run 5
False Positives: 1
False Negatives: 0
-----

Average over 5 runs:
Avg False Positives: 0.2
Avg False Negatives: 0.4

Running LSH for 200 hashes | r=5, b=40
Run 1
False Positives: 0
False Negatives: 0
-----
Run 2
False Positives: 0
False Negatives: 0
-----
Run 3
False Positives: 0
False Negatives: 0
-----
Run 4
False Positives: 0
False Negatives: 1
-----
Run 5
False Positives: 0
False Negati